# RoBERTa Transfer Learning — Category Classifier

> **Sanitized release.** This notebook is a faithful copy of the original working
> notebook from a federal document-classification project, edited for public
> sharing: cell outputs are stripped, file paths and system names are
> generalized, and the ten real business categories are renamed `CAT-A` … `CAT-J`.
> Cells that were duplicated once per category in the original are collapsed to
> one or two representative copies, marked with a note. Nothing else about the
> structure or approach has been changed.


Adapted from the sentiment-analysis fine-tuning tutorial by Venelin Valkov:
https://github.com/curiousily/Getting-Things-Done-with-Pytorch/blob/master/08.sentiment-analysis-with-bert.ipynb

RoBERTa: https://arxiv.org/abs/1907.11692

One notebook of this shape was run per category (10 total). This copy is the
representative CAT-A run.

In [ ]:
#ref: potential optimizer https://towardsdatascience.com/advanced-techniques-for-fine-tuning-transformers-82e4e61e1
import transformers
from transformers import RobertaModel, RobertaTokenizer, AdamW, get_linear_schedule_with_warmup

import os as os
import sys as sys
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import rc
from pylab import rcParams
from torch import nn, optim
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

#set graph config
%matplotlib inline
%config InlineBackend.figure_format='retina'
sns.set(style='whitegrid', palette='muted', font_scale=1.2)
rcParams['figure.figsize'] = 12, 8

#set seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

#check gpu
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#device = torch.device("cpu")  #fallback used for configs that exceeded the 8 GB GPU
device

In [ ]:
#load df from the categorical assignment step of the Logistic Regression notebook
df = pd.read_csv('model_recordflag_doctype_onespace_CAT-A_general.csv')

#split record-number docs (deploy set) from labeled docs (training pool)
df_record = df[df['doc_type'] == 'RECORD']
df_labeled = df[df['doc_type'] != 'RECORD']

#drop df from memory
df = None

In [ ]:
#drop unnecessary columns
df_labeled = df_labeled.drop(['document_id', 'record_flag', 'directory', 'file_name', 'old_ocr', 'doc_type'], axis=1)

#rename cols for pytorch stuff
#('content'/'sentiment' naming inherited from the sentiment tutorial this was adapted from)
df_labeled.columns = ['content', 'sentiment']

#inspect  (representative category: (64706, 2))
df_labeled.shape

In [ ]:
#class balance  (representative category: 51805 negative / 12901 positive — roughly 80/20)
df_labeled.sentiment.value_counts()

In [ ]:
#identify model — a publicly available pre-trained RoBERTa-large checkpoint
PRE_TRAINED_MODEL_NAME = 'siebert/sentiment-roberta-large-english'

#load tokenizer
tokenizer = RobertaTokenizer.from_pretrained(PRE_TRAINED_MODEL_NAME)

In [ ]:
#as a result of trial and error, due to gpu memory limitations, we need to cap
#the tensor length well below RoBERTa's 512-token window.
#MAX_LEN = 255

MAX_LEN = 250

In [ ]:
#let's format our dataset to PyTorch dataset.
class DocumentDataset(Dataset):

  def __init__(self, documents, targets, tokenizer, max_len):
    self.documents = documents
    self.targets = targets
    self.tokenizer = tokenizer
    self.max_len = max_len

  def __len__(self):
    return len(self.documents)

  def __getitem__(self, item):
    document = str(self.documents[item])
    target = self.targets[item]

    encoding = self.tokenizer.encode_plus(
      document,
      add_special_tokens=True,
      max_length=self.max_len,
      return_token_type_ids=False,
      truncation=True,
      padding='max_length',
      return_attention_mask=True,
      return_tensors='pt',
    )

    return {
      'document_text': document,
      'input_ids': encoding['input_ids'].flatten(),
      'attention_mask': encoding['attention_mask'].flatten(),
      'targets': torch.tensor(target, dtype=torch.long)
    }

In [ ]:
#test-train-split training data  (representative: 58235 / 3235 / 3236)
df_train, df_test = train_test_split(df_labeled, test_size=0.1, random_state=RANDOM_SEED)
df_val, df_test = train_test_split(df_test, test_size=0.5, random_state=RANDOM_SEED)

df_train.shape, df_val.shape, df_test.shape

In [ ]:
#data loader
def create_data_loader(df, tokenizer, max_len, batch_size):
  ds = DocumentDataset(
    documents=df.content.to_numpy(),
    targets=df.sentiment.to_numpy(),
    tokenizer=tokenizer,
    max_len=max_len
  )

  return DataLoader(
    ds,
    batch_size=batch_size,
    num_workers=0 #when running pytorch in windows, not linux, you need to set this to zero or it crashes.
  )               #https://github.com/pytorch/pytorch/issues/2341

In [ ]:
#set batch size and loader values (batch size found by trial and error against GPU memory)
BATCH_SIZE = 12

train_data_loader = create_data_loader(df_train, tokenizer, MAX_LEN, BATCH_SIZE)
val_data_loader = create_data_loader(df_val, tokenizer, MAX_LEN, BATCH_SIZE)
test_data_loader = create_data_loader(df_test, tokenizer, MAX_LEN, BATCH_SIZE)

In [ ]:
#classifier head on top of the pre-trained encoder
class CategoryClassifier(nn.Module):

  def __init__(self, n_classes):
    super(CategoryClassifier, self).__init__()
    self.roberta = RobertaModel.from_pretrained(PRE_TRAINED_MODEL_NAME, return_dict=False)
    self.drop = nn.Dropout(p=0.1)
    self.out = nn.Linear(self.roberta.config.hidden_size, n_classes)

  def forward(self, input_ids, attention_mask):
    _, pooled_output = self.roberta(
      input_ids=input_ids,
      attention_mask=attention_mask
    )
    output = self.drop(pooled_output)
    return self.out(output)

In [ ]:
#make an instance of the above class and move it to the device
class_names = ['NOT-CAT-A', 'CAT-A']
model = CategoryClassifier(len(class_names))
model = model.to(device)

In [ ]:
EPOCHS = 3

optimizer = AdamW(model.parameters(), lr=1e-6, correct_bias=True, weight_decay=0.01)
total_steps = len(train_data_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
  optimizer,
  num_warmup_steps=10000,
  num_training_steps=total_steps
)

loss_fn = nn.CrossEntropyLoss().to(device)

In [ ]:
#set training function
def train_epoch(
  model,
  data_loader,
  loss_fn,
  optimizer,
  device,
  scheduler,
  n_examples
):
  model = model.train()

  losses = []
  correct_predictions = 0

  for d in tqdm(data_loader):
    input_ids = d["input_ids"].to(device)
    attention_mask = d["attention_mask"].to(device)
    targets = d["targets"].to(device)

    outputs = model(
      input_ids=input_ids,
      attention_mask=attention_mask
    )

    _, preds = torch.max(outputs, dim=1)
    loss = loss_fn(outputs, targets)

    correct_predictions += torch.sum(preds == targets)
    losses.append(loss.item())

    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

  return correct_predictions.double() / n_examples, np.mean(losses)

In [ ]:
# set eval function
def eval_model(model, data_loader, loss_fn, device, n_examples):
  model = model.eval()

  losses = []
  correct_predictions = 0

  with torch.no_grad():
    for d in tqdm(data_loader):
      input_ids = d["input_ids"].to(device)
      attention_mask = d["attention_mask"].to(device)
      targets = d["targets"].to(device)

      outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
      )
      _, preds = torch.max(outputs, dim=1)

      loss = loss_fn(outputs, targets)

      correct_predictions += torch.sum(preds == targets)
      losses.append(loss.item())

  return correct_predictions.double() / n_examples, np.mean(losses)

In [ ]:
#clear gpu cache before training
torch.cuda.empty_cache()

In [ ]:
# execute training
history = defaultdict(list)
best_accuracy = 0

for epoch in range(EPOCHS):

  print(f'Epoch {epoch + 1}/{EPOCHS}')
  print('-' * 10)

  train_acc, train_loss = train_epoch(
    model,
    train_data_loader,
    loss_fn,
    optimizer,
    device,
    scheduler,
    len(df_train)
  )

  print(f'Train loss {train_loss} accuracy {train_acc}')

  val_acc, val_loss = eval_model(
    model,
    val_data_loader,
    loss_fn,
    device,
    len(df_val)
  )

  print(f'Val   loss {val_loss} accuracy {val_acc}')
  print()

  history['train_acc'].append(train_acc)
  history['train_loss'].append(train_loss)
  history['val_acc'].append(val_acc)
  history['val_loss'].append(val_loss)

  if val_acc > best_accuracy:
    torch.save(model.state_dict(), 'results_roberta_len250_batch12_lr1e-6_decay0-01_CAT-A.bin')
    best_accuracy = val_acc

**Documented results from the original run (outputs stripped in this release):**
after the first epoch, train accuracy 0.895 / loss 0.284 and **validation
accuracy 0.961** / loss 0.163. Configurations that exceeded the 8 GB GPU had to
fall back to CPU, where one epoch over ~58k training documents took **~35
hours** — every experiment had to be planned, not casually re-run.

In [ ]:
#gather training results
df_history = pd.DataFrame.from_dict(history, orient='index')
df_history_t = df_history.transpose()

#make figure data
train_acc = [a.detach().cpu().numpy() for a in df_history_t['train_acc'].tolist()]
val_acc = [a.item() for a in df_history_t['val_acc'].tolist()]

# inspect training results
plt.plot(train_acc, label='train accuracy')
plt.plot(val_acc, label='validation accuracy')

plt.title('Training history')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.ylim([0.85, 1])

# Evaluation on record-number files

In [ ]:
import pandas as pd
#load df from previous categorical assignment step
df = pd.read_csv('model_recordflag_doctype_onespace_CAT-A_general.csv')

#prep set for processing
document_id = df['document_id'].tolist()
pdf_dir = df['directory'].tolist()
new_ocr = df['new_ocr'].tolist()

In [ ]:
#load trained model from the above
model = CategoryClassifier(len(class_names))
model.load_state_dict(torch.load('results_roberta_len250_batch12_lr1e-6_decay0-01_CAT-A.bin'))
model = model.to(device)

test_acc, _ = eval_model(
  model,
  test_data_loader,
  loss_fn,
  device,
  len(df_test)
)

test_acc.item()

In [ ]:
def get_predictions(model, data_loader):
  model = model.eval()

  document_texts = []
  predictions = []
  prediction_probs = []
  real_values = []

  with torch.no_grad():
    for d in tqdm(data_loader):

      texts = d["document_text"]
      input_ids = d["input_ids"].to(device)
      attention_mask = d["attention_mask"].to(device)
      targets = d["targets"].to(device)

      outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
      )
      _, preds = torch.max(outputs, dim=1)

      probs = F.softmax(outputs, dim=1)

      document_texts.extend(texts)
      predictions.extend(preds)
      prediction_probs.extend(probs)
      real_values.extend(targets)

  predictions = torch.stack(predictions).cpu()
  prediction_probs = torch.stack(prediction_probs).cpu()
  real_values = torch.stack(real_values).cpu()
  return document_texts, predictions, prediction_probs, real_values

In [ ]:
y_document_texts, y_pred, y_pred_probs, y_test = get_predictions(model, test_data_loader)
print(classification_report(y_test, y_pred, target_names=class_names))

# Predicting on Raw Text

Score every document in the corpus with the fine-tuned model — the transformer half of the 20-classifier batch-inference pass.

In [ ]:
# pass each document through the fine-tuned model
temp_document_id = []
prediction_lst = []

for id, doc in tqdm(zip(document_id, new_ocr)):
    encoded_document = tokenizer.encode_plus(doc, max_length=MAX_LEN,
                                             add_special_tokens=True,
                                             return_token_type_ids=False,
                                             truncation=True,
                                             padding='max_length',
                                             return_attention_mask=True,
                                             return_tensors='pt',)

    input_ids = encoded_document['input_ids'].to(device)
    attention_mask = encoded_document['attention_mask'].to(device)
    output = model(input_ids, attention_mask)
    _, prediction = torch.max(output, dim=1)

    temp_document_id.append(id)
    prediction_lst.append(class_names[prediction])

In [ ]:
#convert prediction_lst from strings to numbers
new_prediction_lst = []

for item in prediction_lst:
    if item == 'CAT-A':
        new_prediction_lst.append(1)
    else:
        new_prediction_lst.append(0)

#move results to pd df
df_pred = pd.DataFrame(list(zip(temp_document_id, new_prediction_lst)), columns=['document_id', 'pred'])

#save as csv
df_pred.to_csv('results_roberta_len250_batch12_lr1e-6_decay0-01_CAT-A.csv', index=False)